In [2]:
# STEP 1: INVENTORY INTEGRATED ANALYTICAL DATASETS

from pathlib import Path
import pandas as pd

# Locate the project folder
project_root = Path.cwd()

# Locate processed analytical outputs
processed_dir = project_root / "data" / "processed"

assert processed_dir.is_dir(), (
    f"Processed data folder not found: {processed_dir}"
)

# Identify all processed CSV files
csv_files = sorted(processed_dir.glob("*.csv"))

# Identify the originating analytical module
def identify_source(filename):

    if filename.startswith("retail_"):
        return "Retail Analytics"

    elif filename.startswith("apple_headphones_"):
        return "eCommerce Behavior"

    elif filename.startswith("demand_sensing_"):
        return "eCommerce Behavior"

    elif filename.startswith("purchase_intent_"):
        return "Purchase Intent"

    elif (
        filename.startswith("airpods_")
        or filename.startswith("model_")
    ):
        return "Historical Demand Forecasting"

    else:
        return "Other / Review"


# Build our data inventory
inventory_records = []

for file in csv_files:

    # Read column names only
    columns = pd.read_csv(
        file,
        nrows=0
    ).columns.tolist()

    inventory_records.append({
        "File_Name": file.name,
        "Source_Module": identify_source(file.name),
        "File_Size_KB": round(
            file.stat().st_size / 1024, 2
        ),
        "Column_Count": len(columns),
        "Columns": ", ".join(columns)
    })

# Create the integrated catalog
data_catalog = pd.DataFrame(inventory_records)

print("===== INTEGRATED DATA CATALOG =====")
print("Processed CSV files:", len(data_catalog))

display(data_catalog)

# Summarize analytical modules
print("\n===== DATASETS BY SOURCE MODULE =====")

display(
    data_catalog
    .groupby("Source_Module")
    .size()
    .reset_index(name="File_Count")
)

print("\n===== STEP 1 COMPLETE =====")


===== INTEGRATED DATA CATALOG =====
Processed CSV files: 16


,File_Name,Source_Module,File_Size_KB,Column_Count,Columns
0,airpods_weekly_2019.csv,Historical Demand Forecasting,0.77,2,"Week_Ending, Units_Sold"
1,apple_headphones_daily_2019.csv,eCommerce Behavior,3.36,9,"Event_Date, Total_Events, Views, Cart_Events, ..."
2,apple_headphones_weekly_2019.csv,eCommerce Behavior,0.85,10,"Week_Ending, Days_Observed, Total_Events, View..."
3,demand_sensing_holdout_predictions_2019.csv,eCommerce Behavior,1.06,5,"Event_Date, Purchases_For_Analysis, Naive_Fore..."
4,demand_sensing_model_comparison_2019.csv,eCommerce Behavior,0.14,4,"Model, MAE, WAPE (%), Bias"
5,model_performance_rolling_2019.csv,Historical Demand Forecasting,0.47,7,"Rank, Model, MAE, WAPE (%), Bias, Forecast_Hor..."
6,model_predictions_rolling_2019.csv,Historical Demand Forecasting,1.59,7,"Week_Ending, Actual_Units, Naive_Forecast, MA4..."
7,purchase_intent_confusion_summary.csv,Purchase Intent,0.13,5,"Model, True_Negatives, False_Positives, False_..."
8,purchase_intent_model_leaderboard.csv,Purchase Intent,0.89,7,"Model, Feature_Group, Accuracy, Precision, Rec..."
9,retail_category_monthly_2019.csv,Retail Analytics,1.63,4,"Month, Category, Units_Sold, Total_Orders"



===== DATASETS BY SOURCE MODULE =====


,Source_Module,File_Count
0,Historical Demand Forecasting,3
1,Purchase Intent,2
2,Retail Analytics,7
3,eCommerce Behavior,4



===== STEP 1 COMPLETE =====


In [4]:
# STEP 2: VALIDATE DATASET QUALITY AND DATE COVERAGE

import pandas as pd

# ------------------------------------------------------------
# PART 1: DEFINE EXPECTED ROW COUNTS
# ------------------------------------------------------------

# Counts verified during our earlier Kaggle analyses.
# Files without an established count will be marked
# as "Not specified" instead of assumed correct.

expected_rows = {
    "airpods_weekly_2019.csv": 51,
    "apple_headphones_daily_2019.csv": 61,
    "apple_headphones_weekly_2019.csv": 9,
    "demand_sensing_holdout_predictions_2019.csv": 17,
    "demand_sensing_model_comparison_2019.csv": 3,
    "purchase_intent_model_leaderboard.csv": 7,
    "purchase_intent_confusion_summary.csv": 2,
    "retail_category_monthly_2019.csv": 48,
    "retail_channel_demand.csv": 2,
    "retail_country_demand.csv": 8,
    "retail_monthly_coverage.csv": 60,
    "retail_monthly_demand.csv": 62,
    "retail_top_products.csv": 15,
    "retail_yearly_demand.csv": 6
}

# ------------------------------------------------------------
# PART 2: IDENTIFY DATE COVERAGE
# ------------------------------------------------------------

def find_date_coverage(df):

    date_columns = [
        "Event_Date",
        "Week_Ending",
        "Order_Date",
        "Month"
    ]

    for column in date_columns:

        if column not in df.columns:
            continue

        values = df[column]

        # Avoid misreading numeric month values as dates.
        if pd.api.types.is_numeric_dtype(values):
            continue

        parsed = pd.to_datetime(
            values,
            errors="coerce"
        )

        if parsed.notna().any():

            return (
                parsed.min().strftime("%Y-%m-%d"),
                parsed.max().strftime("%Y-%m-%d")
            )

    # Annual summaries have only a year field.
    if "Year" in df.columns:

        years = pd.to_numeric(
            df["Year"],
            errors="coerce"
        ).dropna()

        if not years.empty:
            return (
                str(int(years.min())),
                str(int(years.max()))
            )

    return ("N/A", "N/A")


# ------------------------------------------------------------
# PART 3: VALIDATE ALL PROCESSED CSV FILES
# ------------------------------------------------------------

quality_records = []

for file in csv_files:

    df = pd.read_csv(file)

    row_count = len(df)
    missing_cells = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    start_date, end_date = find_date_coverage(df)

    expected = expected_rows.get(file.name)

    if expected is None:
        row_check = "Not specified"
    elif row_count == expected:
        row_check = "PASS"
    else:
        row_check = "REVIEW"

    quality_records.append({
        "File_Name": file.name,
        "Source_Module": identify_source(file.name),
        "Rows": row_count,
        "Columns": len(df.columns),
        "Missing_Cells": missing_cells,
        "Duplicate_Rows": duplicate_rows,
        "Start_Date": start_date,
        "End_Date": end_date,
        "Row_Check": row_check
    })


# ------------------------------------------------------------
# PART 4: DISPLAY QUALITY REPORT
# ------------------------------------------------------------

quality_report = pd.DataFrame(quality_records)

print("===== INTEGRATED DATA QUALITY REPORT =====")

display(quality_report)

print("\n===== QUALITY SUMMARY =====")

print("Files inspected:", len(quality_report))

print(
    "Files with missing cells:",
    int((quality_report["Missing_Cells"] > 0).sum())
)

print(
    "Files with duplicate rows:",
    int((quality_report["Duplicate_Rows"] > 0).sum())
)

print(
    "Expected row counts passed:",
    int((quality_report["Row_Check"] == "PASS").sum())
)

print(
    "Files requiring row-count review:",
    int((quality_report["Row_Check"] == "REVIEW").sum())
)

print("\n===== STEP 2 COMPLETE =====")


===== INTEGRATED DATA QUALITY REPORT =====


,File_Name,Source_Module,Rows,Columns,Missing_Cells,Duplicate_Rows,Start_Date,End_Date,Row_Check
0,airpods_weekly_2019.csv,Historical Demand Forecasting,51,2,0,0,2019-01-13,2019-12-29,PASS
1,apple_headphones_daily_2019.csv,eCommerce Behavior,61,9,1,0,2019-10-01,2019-11-30,PASS
2,apple_headphones_weekly_2019.csv,eCommerce Behavior,9,10,0,0,2019-10-06,2019-12-01,PASS
3,demand_sensing_holdout_predictions_2019.csv,eCommerce Behavior,17,5,0,0,2019-11-10,2019-11-30,PASS
4,demand_sensing_model_comparison_2019.csv,eCommerce Behavior,3,4,0,0,N/A,N/A,PASS
5,model_performance_rolling_2019.csv,Historical Demand Forecasting,5,7,0,0,N/A,N/A,Not specified
6,model_predictions_rolling_2019.csv,Historical Demand Forecasting,19,7,0,0,2019-06-02,2019-10-06,Not specified
7,purchase_intent_confusion_summary.csv,Purchase Intent,2,5,0,0,N/A,N/A,PASS
8,purchase_intent_model_leaderboard.csv,Purchase Intent,7,7,0,0,N/A,N/A,PASS
9,retail_category_monthly_2019.csv,Retail Analytics,48,4,0,0,2019-01-01,2019-12-01,PASS



===== QUALITY SUMMARY =====
Files inspected: 16
Files with missing cells: 2
Files with duplicate rows: 0
Expected row counts passed: 14
Files requiring row-count review: 0

===== STEP 2 COMPLETE =====


In [5]:
# STEP 2B: INVESTIGATE MISSING VALUES

import pandas as pd

# Files identified during Step 2
files_to_investigate = [
    "apple_headphones_daily_2019.csv",
    "retail_monthly_demand.csv"
]

for filename in files_to_investigate:

    file_path = processed_dir / filename

    df = pd.read_csv(file_path)

    print("\n" + "=" * 60)
    print("INVESTIGATING:", filename)
    print("=" * 60)

    # --------------------------------------------------------
    # PART 1: IDENTIFY COLUMNS WITH MISSING VALUES
    # --------------------------------------------------------

    missing_summary = (
        df.isna()
        .sum()
        .reset_index()
    )

    missing_summary.columns = [
        "Column",
        "Missing_Count"
    ]

    missing_summary = missing_summary[
        missing_summary["Missing_Count"] > 0
    ]

    print("\n===== COLUMNS WITH MISSING VALUES =====")
    display(missing_summary)

    # --------------------------------------------------------
    # PART 2: DISPLAY AFFECTED RECORDS
    # --------------------------------------------------------

    affected_rows = df[
        df.isna().any(axis=1)
    ]

    print("\n===== RECORDS CONTAINING MISSING VALUES =====")
    print("Affected records:", len(affected_rows))

    display(affected_rows)

    # --------------------------------------------------------
    # PART 3: IDENTIFY DATA TYPES
    # --------------------------------------------------------

    print("\n===== DATA TYPES OF AFFECTED COLUMNS =====")

    for col in missing_summary["Column"]:

        print(
            f"{col}: {df[col].dtype}"
        )

    # --------------------------------------------------------
    # PART 4: CALCULATE MISSING-VALUE RATE
    # --------------------------------------------------------

    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())

    missing_rate = (
        missing_cells / total_cells * 100
        if total_cells > 0 else 0
    )

    print(
        "\nMissing cell rate:",
        round(missing_rate, 3), "%"
    )


print("\n===== STEP 2B COMPLETE =====")



INVESTIGATING: apple_headphones_daily_2019.csv

===== COLUMNS WITH MISSING VALUES =====


,Column,Missing_Count
8,Purchases_For_Analysis,1



===== RECORDS CONTAINING MISSING VALUES =====
Affected records: 1


,Event_Date,Total_Events,Views,Cart_Events,Purchases,Unique_Products,Month,Purchase_Anomaly_Flag,Purchases_For_Analysis
45,2019-11-15,76366,62857,13509,0,12,2019-11,True,NaN



===== DATA TYPES OF AFFECTED COLUMNS =====
Purchases_For_Analysis: float64

Missing cell rate: 0.182 %

INVESTIGATING: retail_monthly_demand.csv

===== COLUMNS WITH MISSING VALUES =====


,Column,Missing_Count
6,Calendar_Days,2
7,Days_With_Sales,2
8,Recorded_Day_Pct,2



===== RECORDS CONTAINING MISSING VALUES =====
Affected records: 2


,Month,Year,Month_Number,Units_Sold,Total_Orders,Unique_Products,Calendar_Days,Days_With_Sales,Recorded_Day_Pct,Coverage_Review_Status
60,2021-01-01,2021,1,1848.0,256,493,NaN,NaN,NaN,2021_Partial_Year_Not_Audited
61,2021-02-01,2021,2,1949.0,242,463,NaN,NaN,NaN,2021_Partial_Year_Not_Audited



===== DATA TYPES OF AFFECTED COLUMNS =====
Calendar_Days: float64
Days_With_Sales: float64
Recorded_Day_Pct: float64

Missing cell rate: 0.968 %

===== STEP 2B COMPLETE =====


In [6]:
# STEP 3: DEFINE INTEGRATED ANALYTICAL ARCHITECTURE

import pandas as pd

# ------------------------------------------------------------
# PART 1: DEFINE DATASET GRANULARITY AND PURPOSE
# ------------------------------------------------------------

dataset_specs = {
    "airpods_weekly_2019.csv": (
        "Week",
        "Historical units sold",
        "Forecasting input"
    ),
    "model_performance_rolling_2019.csv": (
        "Forecast model",
        "Rolling backtest performance",
        "Model evaluation"
    ),
    "model_predictions_rolling_2019.csv": (
        "Week",
        "Actual versus predicted demand",
        "Forecast validation"
    ),
    "apple_headphones_daily_2019.csv": (
        "Day",
        "Views, carts, purchase events",
        "Demand signals"
    ),
    "apple_headphones_weekly_2019.csv": (
        "Week",
        "Aggregated behavioral events",
        "Demand signals"
    ),
    "demand_sensing_model_comparison_2019.csv": (
        "Forecast model",
        "Behavioral model comparison",
        "Model evaluation"
    ),
    "demand_sensing_holdout_predictions_2019.csv": (
        "Day",
        "Actual versus predicted purchase events",
        "Forecast validation"
    ),
    "retail_monthly_demand.csv": (
        "Month",
        "Retail demand and coverage",
        "Historical demand intelligence"
    ),
    "retail_yearly_demand.csv": (
        "Year",
        "Annual retail demand",
        "Historical demand intelligence"
    ),
    "retail_category_monthly_2019.csv": (
        "Month x category",
        "Retail category demand",
        "Product mix analysis"
    ),
    "retail_country_demand.csv": (
        "Country",
        "Geographic demand",
        "Regional analysis"
    ),
    "retail_channel_demand.csv": (
        "Sales channel",
        "Online versus physical sales",
        "Channel analysis"
    ),
    "retail_top_products.csv": (
        "Product",
        "Top-selling products",
        "Product analysis"
    ),
    "retail_monthly_coverage.csv": (
        "Month",
        "Recorded sales-day coverage",
        "Data quality"
    ),
    "purchase_intent_model_leaderboard.csv": (
        "Model x feature group",
        "Classification performance",
        "Purchase intent research"
    ),
    "purchase_intent_confusion_summary.csv": (
        "Classification model",
        "Prediction outcome counts",
        "Model evaluation"
    )
}

# ------------------------------------------------------------
# PART 2: BUILD THE ARCHITECTURE CATALOG
# ------------------------------------------------------------

architecture_catalog = data_catalog[
    ["File_Name", "Source_Module"]
].copy()

architecture_catalog["Grain"] = (
    architecture_catalog["File_Name"]
    .map(lambda f: dataset_specs[f][0])
)

architecture_catalog["Business_Meaning"] = (
    architecture_catalog["File_Name"]
    .map(lambda f: dataset_specs[f][1])
)

architecture_catalog["Analytical_Role"] = (
    architecture_catalog["File_Name"]
    .map(lambda f: dataset_specs[f][2])
)

print("===== INTEGRATED ANALYTICAL ARCHITECTURE =====")

display(architecture_catalog)

# ------------------------------------------------------------
# PART 3: DEFINE KEY BUSINESS METRICS
# ------------------------------------------------------------

kpi_dictionary = pd.DataFrame([
    {
        "KPI": "Weekly Units Sold",
        "Module": "Historical Demand Forecasting",
        "Definition": "Observed units sold during a week",
        "Unit": "Units"
    },
    {
        "KPI": "Forecast WAPE",
        "Module": "Historical Demand Forecasting",
        "Definition": "Sum absolute errors / sum actual demand",
        "Unit": "%"
    },
    {
        "KPI": "Forecast Bias",
        "Module": "Historical Demand Forecasting",
        "Definition": "Mean predicted demand minus actual demand",
        "Unit": "Units or events"
    },
    {
        "KPI": "Product Views",
        "Module": "eCommerce Behavior",
        "Definition": "Recorded product-view events",
        "Unit": "Events"
    },
    {
        "KPI": "Cart Activity",
        "Module": "eCommerce Behavior",
        "Definition": "Recorded add-to-cart events",
        "Unit": "Events"
    },
    {
        "KPI": "Purchase Events",
        "Module": "eCommerce Behavior",
        "Definition": "Recorded purchase events, not verified units",
        "Unit": "Events"
    },
    {
        "KPI": "Retail Units Sold",
        "Module": "Retail Analytics",
        "Definition": "Aggregated historical retail units",
        "Unit": "Units"
    },
    {
        "KPI": "Purchase Intent ROC-AUC",
        "Module": "Purchase Intent",
        "Definition": "Classification ranking performance",
        "Unit": "Score (0-1)"
    }
])

print("\n===== BUSINESS KPI DICTIONARY =====")

display(kpi_dictionary)

# ------------------------------------------------------------
# PART 4: VALIDATE ARCHITECTURE COMPLETENESS
# ------------------------------------------------------------

actual_files = set(data_catalog["File_Name"])
specified_files = set(dataset_specs.keys())

assert actual_files == specified_files, (
    "Architecture catalog does not match available files."
)

assert architecture_catalog.notna().all().all()

print("\n===== ARCHITECTURE VALIDATION =====")

print("Files accounted for:", len(architecture_catalog))
print("Business KPIs defined:", len(kpi_dictionary))
print("Missing architecture definitions: 0")

print("\n===== STEP 3 COMPLETE =====")


===== INTEGRATED ANALYTICAL ARCHITECTURE =====


,File_Name,Source_Module,Grain,Business_Meaning,Analytical_Role
0,airpods_weekly_2019.csv,Historical Demand Forecasting,Week,Historical units sold,Forecasting input
1,apple_headphones_daily_2019.csv,eCommerce Behavior,Day,"Views, carts, purchase events",Demand signals
2,apple_headphones_weekly_2019.csv,eCommerce Behavior,Week,Aggregated behavioral events,Demand signals
3,demand_sensing_holdout_predictions_2019.csv,eCommerce Behavior,Day,Actual versus predicted purchase events,Forecast validation
4,demand_sensing_model_comparison_2019.csv,eCommerce Behavior,Forecast model,Behavioral model comparison,Model evaluation
5,model_performance_rolling_2019.csv,Historical Demand Forecasting,Forecast model,Rolling backtest performance,Model evaluation
6,model_predictions_rolling_2019.csv,Historical Demand Forecasting,Week,Actual versus predicted demand,Forecast validation
7,purchase_intent_confusion_summary.csv,Purchase Intent,Classification model,Prediction outcome counts,Model evaluation
8,purchase_intent_model_leaderboard.csv,Purchase Intent,Model x feature group,Classification performance,Purchase intent research
9,retail_category_monthly_2019.csv,Retail Analytics,Month x category,Retail category demand,Product mix analysis



===== BUSINESS KPI DICTIONARY =====


,KPI,Module,Definition,Unit
0,Weekly Units Sold,Historical Demand Forecasting,Observed units sold during a week,Units
1,Forecast WAPE,Historical Demand Forecasting,Sum absolute errors / sum actual demand,%
2,Forecast Bias,Historical Demand Forecasting,Mean predicted demand minus actual demand,Units or events
3,Product Views,eCommerce Behavior,Recorded product-view events,Events
4,Cart Activity,eCommerce Behavior,Recorded add-to-cart events,Events
5,Purchase Events,eCommerce Behavior,"Recorded purchase events, not verified units",Events
6,Retail Units Sold,Retail Analytics,Aggregated historical retail units,Units
7,Purchase Intent ROC-AUC,Purchase Intent,Classification ranking performance,Score (0-1)



===== ARCHITECTURE VALIDATION =====
Files accounted for: 16
Business KPIs defined: 8
Missing architecture definitions: 0

===== STEP 3 COMPLETE =====


In [9]:
# STEP 4: BUILD PERSISTENT DUCKDB ANALYTICAL DATABASE

from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: CLOSE ANY PREVIOUS DATABASE CONNECTION
# ------------------------------------------------------------

if "con" in globals():
    try:
        con.close()
    except Exception:
        pass


# ------------------------------------------------------------
# PART 2: DEFINE DATABASE LOCATION
# ------------------------------------------------------------

database_dir = project_root / "data" / "warehouse"

database_dir.mkdir(
    parents=True,
    exist_ok=True
)

database_path = (
    database_dir / "demand_intelligence.duckdb"
)

print("===== DATABASE LOCATION =====")
print(database_path)


# ------------------------------------------------------------
# PART 3: MAP DATA SOURCES TO SQL SCHEMAS
# ------------------------------------------------------------

schema_mapping = {
    "Historical Demand Forecasting": "demand",
    "eCommerce Behavior": "ecommerce",
    "Retail Analytics": "retail",
    "Purchase Intent": "intent"
}

schema_names = [
    "demand",
    "ecommerce",
    "retail",
    "intent",
    "metadata"
]


# ------------------------------------------------------------
# PART 4: CONNECT TO PERSISTENT DUCKDB DATABASE
# ------------------------------------------------------------

con = duckdb.connect(
    str(database_path)
)

print("\n===== DATABASE CONNECTION =====")
print("Connection established successfully.")


try:

    # --------------------------------------------------------
    # PART 5: CREATE DATABASE SCHEMAS
    # --------------------------------------------------------

    for schema in schema_names:

        con.execute(
            f"CREATE SCHEMA IF NOT EXISTS {schema}"
        ).fetchall()

    print("SQL schemas created:", len(schema_names))


    # --------------------------------------------------------
    # PART 6: LOAD ALL PROCESSED CSV FILES
    # --------------------------------------------------------

    load_results = []

    for file in csv_files:

        # Identify the dataset's source module
        source_module = identify_source(
            file.name
        )

        assert source_module in schema_mapping, (
            f"Unknown source module: {file.name}"
        )

        # Determine destination schema and table
        schema = schema_mapping[source_module]

        table_name = file.stem

        assert table_name.isidentifier(), (
            f"Invalid SQL table name: {table_name}"
        )

        # Load processed CSV into Pandas
        df = pd.read_csv(file)

        # Register DataFrame with DuckDB
        con.register(
            "incoming_data",
            df
        )

        try:

            # Create or replace persistent SQL table
            con.execute(
                f"""
                CREATE OR REPLACE TABLE
                    {schema}.{table_name}
                AS
                SELECT *
                FROM incoming_data
                """
            ).fetchall()

        finally:

            # Remove temporary DataFrame registration
            con.unregister("incoming_data")


        # ----------------------------------------------------
        # VALIDATE PERSISTED ROW COUNT
        # ----------------------------------------------------

        # IMPORTANT: fetchall() fully consumes the SQL result.

        sql_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM {schema}.{table_name}
            """
        ).fetchall()[0][0]

        csv_count = len(df)

        assert sql_count == csv_count, (
            f"Row count mismatch: {file.name}"
        )

        # Store load results
        load_results.append({
            "Schema": schema,
            "Table": table_name,
            "CSV_Rows": csv_count,
            "SQL_Rows": sql_count,
            "Status": "PASS"
        })

        print(
            f"Loaded: {schema}.{table_name} "
            f"({sql_count} rows)"
        )


    # --------------------------------------------------------
    # PART 7: CREATE METADATA TABLES
    # --------------------------------------------------------

    metadata_tables = {
        "architecture_catalog": architecture_catalog,
        "data_quality_report": quality_report,
        "kpi_dictionary": kpi_dictionary
    }

    print("\n===== LOADING METADATA TABLES =====")

    for table_name, dataframe in metadata_tables.items():

        # Register metadata DataFrame
        con.register(
            "incoming_metadata",
            dataframe
        )

        try:

            # Create persistent metadata table
            con.execute(
                f"""
                CREATE OR REPLACE TABLE
                    metadata.{table_name}
                AS
                SELECT *
                FROM incoming_metadata
                """
            ).fetchall()

        finally:

            con.unregister("incoming_metadata")


        # ----------------------------------------------------
        # VALIDATE METADATA ROW COUNT
        # ----------------------------------------------------

        metadata_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM metadata.{table_name}
            """
        ).fetchall()[0][0]

        assert metadata_count == len(dataframe), (
            f"Metadata row count mismatch: {table_name}"
        )

        print(
            f"Loaded: metadata.{table_name} "
            f"({metadata_count} rows)"
        )


    # --------------------------------------------------------
    # PART 8: CREATE DATABASE LOAD REPORT
    # --------------------------------------------------------

    database_load_report = pd.DataFrame(
        load_results
    )

    print("\n===== DATABASE LOAD RESULTS =====")

    display(database_load_report)


    # --------------------------------------------------------
    # PART 9: SUMMARIZE DATABASE TABLES
    # --------------------------------------------------------

    schema_summary = (
        database_load_report
        .groupby("Schema")
        .agg(
            Tables=("Table", "count"),
            Total_Rows=("SQL_Rows", "sum")
        )
        .reset_index()
    )

    print("\n===== TABLE COUNTS BY SCHEMA =====")

    display(schema_summary)


    # --------------------------------------------------------
    # PART 10: VALIDATE DATABASE STRUCTURE
    # --------------------------------------------------------

    all_tables = con.execute(
        """
        SELECT
            table_schema,
            table_name
        FROM information_schema.tables
        WHERE table_schema IN (
            'demand',
            'ecommerce',
            'retail',
            'intent',
            'metadata'
        )
        AND table_type = 'BASE TABLE'
        ORDER BY
            table_schema,
            table_name
        """
    ).fetchall()

    # 16 processed analytical datasets
    assert len(database_load_report) == 16, (
        "Expected 16 analytical tables."
    )

    # All CSV row counts must match
    assert (
        database_load_report["Status"] == "PASS"
    ).all()

    # 16 analytical tables + 3 metadata tables
    assert len(all_tables) == 19, (
        f"Expected 19 SQL tables, found {len(all_tables)}."
    )


    # --------------------------------------------------------
    # PART 11: VALIDATE SCHEMA DISTRIBUTION
    # --------------------------------------------------------

    expected_schema_counts = {
        "demand": 3,
        "ecommerce": 4,
        "retail": 7,
        "intent": 2,
        "metadata": 3
    }

    actual_schema_counts = {}

    for schema, table_name in all_tables:

        actual_schema_counts[schema] = (
            actual_schema_counts.get(schema, 0) + 1
        )

    assert actual_schema_counts == expected_schema_counts, (
        "Unexpected number of tables in one or more schemas."
    )


    # --------------------------------------------------------
    # PART 12: FINAL DATABASE VALIDATION
    # --------------------------------------------------------

    print("\n===== DATABASE VALIDATION =====")

    print(
        "Analytical tables:",
        len(database_load_report)
    )

    print(
        "Metadata tables:",
        len(metadata_tables)
    )

    print(
        "Total SQL tables:",
        len(all_tables)
    )

    print(
        "All CSV row counts matched:",
        True
    )

    print(
        "Database file exists:",
        database_path.is_file()
    )

    print("\n===== STEP 4 COMPLETE =====")


finally:

    # --------------------------------------------------------
    # PART 13: CLOSE DATABASE CONNECTION
    # --------------------------------------------------------

    con.close()

    print("\nDatabase connection closed.")


===== DATABASE LOCATION =====
/Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence.duckdb

===== DATABASE CONNECTION =====
Connection established successfully.
SQL schemas created: 5
Loaded: demand.airpods_weekly_2019 (51 rows)
Loaded: ecommerce.apple_headphones_daily_2019 (61 rows)
Loaded: ecommerce.apple_headphones_weekly_2019 (9 rows)
Loaded: ecommerce.demand_sensing_holdout_predictions_2019 (17 rows)
Loaded: ecommerce.demand_sensing_model_comparison_2019 (3 rows)
Loaded: demand.model_performance_rolling_2019 (5 rows)
Loaded: demand.model_predictions_rolling_2019 (19 rows)
Loaded: intent.purchase_intent_confusion_summary (2 rows)
Loaded: intent.purchase_intent_model_leaderboard (7 rows)
Loaded: retail.retail_category_monthly_2019 (48 rows)
Loaded: retail.retail_channel_demand (2 rows)
Loaded: retail.retail_country_demand (8 rows)
Loaded: retail.retail_monthly_coverage (60 rows)
Loaded: retail.retail_monthly_demand (62 r

,Schema,Table,CSV_Rows,SQL_Rows,Status
0,demand,airpods_weekly_2019,51,51,PASS
1,ecommerce,apple_headphones_daily_2019,61,61,PASS
2,ecommerce,apple_headphones_weekly_2019,9,9,PASS
3,ecommerce,demand_sensing_holdout_predictions_2019,17,17,PASS
4,ecommerce,demand_sensing_model_comparison_2019,3,3,PASS
5,demand,model_performance_rolling_2019,5,5,PASS
6,demand,model_predictions_rolling_2019,19,19,PASS
7,intent,purchase_intent_confusion_summary,2,2,PASS
8,intent,purchase_intent_model_leaderboard,7,7,PASS
9,retail,retail_category_monthly_2019,48,48,PASS



===== TABLE COUNTS BY SCHEMA =====


,Schema,Tables,Total_Rows
0,demand,3,75
1,ecommerce,4,90
2,intent,2,9
3,retail,7,201



===== DATABASE VALIDATION =====
Analytical tables: 16
Metadata tables: 3
Total SQL tables: 19
All CSV row counts matched: True
Database file exists: True

===== STEP 4 COMPLETE =====

Database connection closed.


In [10]:
# STEP 5: CREATE REUSABLE SQL REPORTING VIEWS

import duckdb
import pandas as pd

# ------------------------------------------------------------
# PART 1: CONNECT TO PERSISTENT DATABASE
# ------------------------------------------------------------

con = duckdb.connect(
    str(database_path)
)

print("===== REPORTING DATABASE CONNECTION =====")
print("Connected to:", database_path)

try:

    # --------------------------------------------------------
    # PART 2: CREATE REPORTING SCHEMA
    # --------------------------------------------------------

    con.execute(
        "CREATE SCHEMA IF NOT EXISTS reporting"
    ).fetchall()

    print("\nReporting schema created.")


    # --------------------------------------------------------
    # PART 3: DEFINE SQL REPORTING VIEWS
    # --------------------------------------------------------

    reporting_views = {

        # A. Historical weekly demand
        "v_airpods_weekly_demand": """
            SELECT
                TRY_CAST(Week_Ending AS DATE)
                    AS Week_Ending,

                CAST(Units_Sold AS DOUBLE)
                    AS Units_Sold

            FROM demand.airpods_weekly_2019
        """,

        # B. eCommerce daily demand signals
        "v_headphone_daily_signals": """
            SELECT
                TRY_CAST(Event_Date AS DATE)
                    AS Event_Date,

                CAST(Views AS BIGINT)
                    AS Views,

                CAST(Cart_Events AS BIGINT)
                    AS Cart_Events,

                CAST(Purchases AS BIGINT)
                    AS Recorded_Purchase_Events,

                CAST(Purchases_For_Analysis AS DOUBLE)
                    AS Purchases_For_Analysis,

                Purchase_Anomaly_Flag,

                CASE
                    WHEN Purchase_Anomaly_Flag = TRUE
                    THEN 'Suspected Recording Anomaly'
                    ELSE 'Unflagged'
                END AS Quality_Status

            FROM ecommerce.apple_headphones_daily_2019
        """,

        # C. Retail monthly demand and coverage
        "v_retail_monthly_demand": """
            SELECT
                TRY_CAST(Month AS DATE)
                    AS Month,

                CAST(Units_Sold AS DOUBLE)
                    AS Units_Sold,

                CAST(Total_Orders AS BIGINT)
                    AS Total_Orders,

                CAST(Unique_Products AS BIGINT)
                    AS Unique_Products,

                Calendar_Days,
                Days_With_Sales,
                Recorded_Day_Pct,
                Coverage_Review_Status

            FROM retail.retail_monthly_demand
        """,

        # D. Forecast performance across independent
        # historical experiments
        "v_forecast_model_performance": """
            SELECT
                'AirPods Weekly Units'
                    AS Experiment,

                'Units'
                    AS Target_Unit,

                Model,

                CAST(MAE AS DOUBLE)
                    AS MAE,

                CAST("WAPE (%)" AS DOUBLE)
                    AS WAPE_Pct,

                CAST(Bias AS DOUBLE)
                    AS Bias

            FROM demand.model_performance_rolling_2019

            UNION ALL

            SELECT
                'Apple Headphone Daily Purchase Events'
                    AS Experiment,

                'Events'
                    AS Target_Unit,

                Model,

                CAST(MAE AS DOUBLE)
                    AS MAE,

                CAST("WAPE (%)" AS DOUBLE)
                    AS WAPE_Pct,

                CAST(Bias AS DOUBLE)
                    AS Bias

            FROM ecommerce.demand_sensing_model_comparison_2019
        """,

        # E. Purchase intent model evaluation
        "v_purchase_intent_models": """
            SELECT
                Model,
                Feature_Group,

                CAST(Accuracy AS DOUBLE)
                    AS Accuracy,

                CAST(Precision AS DOUBLE)
                    AS Precision,

                CAST(Recall AS DOUBLE)
                    AS Recall,

                CAST(F1 AS DOUBLE)
                    AS F1,

                CAST(ROC_AUC AS DOUBLE)
                    AS ROC_AUC

            FROM intent.purchase_intent_model_leaderboard
        """
    }


    # --------------------------------------------------------
    # PART 4: CREATE THE FIVE PERSISTENT VIEWS
    # --------------------------------------------------------

    for view_name, sql_query in reporting_views.items():

        con.execute(
            f"""
            CREATE OR REPLACE VIEW
                reporting.{view_name}
            AS
            {sql_query}
            """
        ).fetchall()

        print("Created:", view_name)


    # --------------------------------------------------------
    # PART 5: VALIDATE REPORTING VIEW ROW COUNTS
    # --------------------------------------------------------

    expected_view_rows = {
        "v_airpods_weekly_demand": 51,
        "v_headphone_daily_signals": 61,
        "v_retail_monthly_demand": 62,
        "v_forecast_model_performance": 8,
        "v_purchase_intent_models": 7
    }

    validation_records = []

    for view_name, expected_count in (
        expected_view_rows.items()
    ):

        row_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM reporting.{view_name}
            """
        ).fetchall()[0][0]

        status = (
            "PASS"
            if row_count == expected_count
            else "REVIEW"
        )

        validation_records.append({
            "View": view_name,
            "Expected_Rows": expected_count,
            "Actual_Rows": row_count,
            "Status": status
        })


    view_validation = pd.DataFrame(
        validation_records
    )

    print("\n===== REPORTING VIEW VALIDATION =====")

    display(view_validation)

    assert (
        view_validation["Status"] == "PASS"
    ).all(), "One or more reporting views need review."


    # --------------------------------------------------------
    # PART 6: CHECK SOURCE QUALITY FLAGS
    # --------------------------------------------------------

    quality_check = con.execute(
        """
        SELECT
            COUNT(*) AS Total_Days,

            SUM(
                CASE
                    WHEN Purchase_Anomaly_Flag = TRUE
                    THEN 1 ELSE 0
                END
            ) AS Flagged_Days,

            SUM(
                CASE
                    WHEN Purchases_For_Analysis IS NULL
                    THEN 1 ELSE 0
                END
            ) AS Excluded_Purchase_Days

        FROM reporting.v_headphone_daily_signals
        """
    ).fetchall()

    quality_summary = pd.DataFrame(
        quality_check,
        columns=[
            "Total_Days",
            "Flagged_Days",
            "Excluded_Purchase_Days"
        ]
    )

    print("\n===== ECOMMERCE QUALITY CHECK =====")

    display(quality_summary)


    # --------------------------------------------------------
    # PART 7: CONFIRM ALL VIEWS EXIST
    # --------------------------------------------------------

    saved_views = con.execute(
        """
        SELECT
            table_name
        FROM information_schema.views
        WHERE table_schema = 'reporting'
        ORDER BY table_name
        """
    ).fetchall()

    print("\n===== PERSISTENT REPORTING VIEWS =====")

    for view in saved_views:
        print(view[0])

    assert len(saved_views) == 5, (
        f"Expected 5 views, found {len(saved_views)}."
    )

    print("\n===== STEP 5 COMPLETE =====")

finally:

    con.close()
    print("\nDatabase connection closed.")


===== REPORTING DATABASE CONNECTION =====
Connected to: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence.duckdb

Reporting schema created.
Created: v_airpods_weekly_demand
Created: v_headphone_daily_signals
Created: v_retail_monthly_demand
Created: v_forecast_model_performance
Created: v_purchase_intent_models

===== REPORTING VIEW VALIDATION =====


,View,Expected_Rows,Actual_Rows,Status
0,v_airpods_weekly_demand,51,51,PASS
1,v_headphone_daily_signals,61,61,PASS
2,v_retail_monthly_demand,62,62,PASS
3,v_forecast_model_performance,8,8,PASS
4,v_purchase_intent_models,7,7,PASS



===== ECOMMERCE QUALITY CHECK =====


,Total_Days,Flagged_Days,Excluded_Purchase_Days
0,61,1,1



===== PERSISTENT REPORTING VIEWS =====
v_airpods_weekly_demand
v_forecast_model_performance
v_headphone_daily_signals
v_purchase_intent_models
v_retail_monthly_demand

===== STEP 5 COMPLETE =====

Database connection closed.


In [11]:
# STEP 6: SQL BUSINESS INTELLIGENCE ANALYSIS

import duckdb
import pandas as pd

# ------------------------------------------------------------
# PART 1: CONNECT TO EXISTING DATABASE
# ------------------------------------------------------------

con = duckdb.connect(
    str(database_path),
    read_only=True
)

print("===== BUSINESS INTELLIGENCE CONNECTION =====")
print("Connected successfully.")


# ------------------------------------------------------------
# PART 2: DEFINE BUSINESS INTELLIGENCE QUERIES
# ------------------------------------------------------------

business_queries = {

    # --------------------------------------------------------
    # QUERY 1: HISTORICAL AIRPODS DEMAND
    # --------------------------------------------------------

    "1. Historical AirPods Demand": """
        SELECT
            COUNT(*) AS Weeks_Observed,

            SUM(Units_Sold) AS Total_Units,

            ROUND(AVG(Units_Sold), 2)
                AS Average_Weekly_Units,

            MIN(Units_Sold) AS Minimum_Weekly_Units,

            MAX(Units_Sold) AS Maximum_Weekly_Units

        FROM reporting.v_airpods_weekly_demand
    """,


    # --------------------------------------------------------
    # QUERY 2: ECOMMERCE BEHAVIOR
    # --------------------------------------------------------

    "2. Apple Headphone Behavioral Signals": """
        SELECT
            COUNT(*) AS Days_Observed,

            SUM(Views) AS Total_Views,

            SUM(Cart_Events) AS Total_Cart_Events,

            SUM(Recorded_Purchase_Events)
                AS Recorded_Purchase_Events,

            SUM(Purchases_For_Analysis)
                AS Purchase_Events_For_Analysis,

            SUM(
                CASE
                    WHEN Purchase_Anomaly_Flag = TRUE
                    THEN 1 ELSE 0
                END
            ) AS Flagged_Days

        FROM reporting.v_headphone_daily_signals
    """,


    # --------------------------------------------------------
    # QUERY 3: RETAIL MONTHLY DEMAND AND COVERAGE
    # --------------------------------------------------------

    "3. Retail Monthly Demand - 2019": """
        SELECT
            Month,

            Units_Sold,

            Total_Orders,

            Unique_Products,

            Calendar_Days,

            Days_With_Sales,

            ROUND(Recorded_Day_Pct, 2)
                AS Recorded_Day_Pct,

            Coverage_Review_Status

        FROM reporting.v_retail_monthly_demand

        WHERE Month >= DATE '2019-01-01'
          AND Month < DATE '2020-01-01'

        ORDER BY Month
    """,


    # --------------------------------------------------------
    # QUERY 4: FORECAST MODEL PERFORMANCE
    # --------------------------------------------------------

    "4. Forecasting Model Performance": """
        SELECT
            Experiment,

            Target_Unit,

            Model,

            ROUND(MAE, 2) AS MAE,

            ROUND(WAPE_Pct, 2) AS WAPE_Pct,

            ROUND(Bias, 2) AS Bias

        FROM reporting.v_forecast_model_performance

        ORDER BY
            Experiment,
            WAPE_Pct ASC
    """,


    # --------------------------------------------------------
    # QUERY 5: PURCHASE INTENT MODEL PERFORMANCE
    # --------------------------------------------------------

    "5. Purchase Intent Classification": """
        SELECT
            Model,

            Feature_Group,

            ROUND(Accuracy, 3) AS Accuracy,

            ROUND(Precision, 3) AS Precision,

            ROUND(Recall, 3) AS Recall,

            ROUND(F1, 3) AS F1,

            ROUND(ROC_AUC, 3) AS ROC_AUC

        FROM reporting.v_purchase_intent_models

        ORDER BY ROC_AUC DESC
    """
}


# ------------------------------------------------------------
# PART 3: EXECUTE ALL BUSINESS QUERIES
# ------------------------------------------------------------

business_results = {}

try:

    for query_name, sql in business_queries.items():

        # Execute SQL and fully consume result.
        result = con.execute(sql)

        rows = result.fetchall()

        column_names = [
            column[0]
            for column in result.description
        ]

        result_df = pd.DataFrame(
            rows,
            columns=column_names
        )

        business_results[query_name] = result_df

        print("\n" + "=" * 65)
        print(query_name.upper())
        print("=" * 65)

        display(result_df)

        print(
            "Records returned:",
            len(result_df)
        )


    # --------------------------------------------------------
    # PART 4: VALIDATE BUSINESS QUERY RESULTS
    # --------------------------------------------------------

    expected_rows = {
        "1. Historical AirPods Demand": 1,
        "2. Apple Headphone Behavioral Signals": 1,
        "3. Retail Monthly Demand - 2019": 12,
        "4. Forecasting Model Performance": 8,
        "5. Purchase Intent Classification": 7
    }

    validation_records = []

    for name, expected in expected_rows.items():

        actual = len(business_results[name])

        validation_records.append({
            "Business_Query": name,
            "Expected_Rows": expected,
            "Actual_Rows": actual,
            "Status": (
                "PASS"
                if actual == expected
                else "REVIEW"
            )
        })

    business_validation = pd.DataFrame(
        validation_records
    )

    print("\n===== BUSINESS QUERY VALIDATION =====")

    display(business_validation)

    assert (
        business_validation["Status"] == "PASS"
    ).all(), "One or more business queries need review."

    print("\n===== STEP 6 COMPLETE =====")

finally:

    con.close()
    print("\nDatabase connection closed safely.")


===== BUSINESS INTELLIGENCE CONNECTION =====
Connected successfully.

1. HISTORICAL AIRPODS DEMAND


,Weeks_Observed,Total_Units,Average_Weekly_Units,Minimum_Weekly_Units,Maximum_Weekly_Units
0,51,15364.0,301.25,175.0,515.0


Records returned: 1

2. APPLE HEADPHONE BEHAVIORAL SIGNALS


,Days_Observed,Total_Views,Total_Cart_Events,Recorded_Purchase_Events,Purchase_Events_For_Analysis,Flagged_Days
0,61,1058017,109366,43377,43377.0,1


Records returned: 1

3. RETAIL MONTHLY DEMAND - 2019


,Month,Units_Sold,Total_Orders,Unique_Products,Calendar_Days,Days_With_Sales,Recorded_Day_Pct,Coverage_Review_Status
0,2019-01-01,6850.0,892,1122,31.0,31.0,100.00,No_Initial_Coverage_Flag
1,2019-02-01,7383.0,967,1130,28.0,28.0,100.00,No_Initial_Coverage_Flag
2,2019-03-01,3095.0,386,660,31.0,19.0,61.29,No_Initial_Coverage_Flag
3,2019-04-01,562.0,75,167,30.0,7.0,23.33,Low_Recorded_Transaction_Days
4,2019-05-01,5966.0,808,1038,31.0,31.0,100.00,No_Initial_Coverage_Flag
5,2019-06-01,5497.0,772,975,30.0,30.0,100.00,No_Initial_Coverage_Flag
6,2019-07-01,5328.0,716,976,31.0,31.0,100.00,No_Initial_Coverage_Flag
7,2019-08-01,5922.0,793,1009,31.0,31.0,100.00,No_Initial_Coverage_Flag
8,2019-09-01,6037.0,781,991,30.0,30.0,100.00,No_Initial_Coverage_Flag
9,2019-10-01,6412.0,842,1076,31.0,31.0,100.00,No_Initial_Coverage_Flag


... (3 outputs left out)

,Model,Feature_Group,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,Satisfaction_Sensitivity,0.687,0.724,0.725,0.724,0.800
1,Logistic Regression,Satisfaction_Sensitivity,0.694,0.721,0.752,0.736,0.779
2,Logistic Regression,Customer_Enriched,0.658,0.677,0.758,0.715,0.665
3,Random Forest,Customer_Enriched,0.682,0.667,0.875,0.757,0.662
4,Random Forest,Product_Context,0.541,0.577,0.706,0.635,0.534
5,Logistic Regression,Product_Context,0.567,0.567,1.000,0.723,0.513
6,Majority Class,Baseline,0.567,0.567,1.000,0.723,0.500


Records returned: 7

===== BUSINESS QUERY VALIDATION =====


,Business_Query,Expected_Rows,Actual_Rows,Status
0,1. Historical AirPods Demand,1,1,PASS
1,2. Apple Headphone Behavioral Signals,1,1,PASS
2,3. Retail Monthly Demand - 2019,12,12,PASS
3,4. Forecasting Model Performance,8,8,PASS
4,5. Purchase Intent Classification,7,7,PASS



===== STEP 6 COMPLETE =====

Database connection closed safely.


In [12]:
# STEP 7: EXPORT DASHBOARD-READY REPORTING DATASETS

from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: DEFINE DASHBOARD EXPORT LOCATION
# ------------------------------------------------------------

dashboard_dir = project_root / "data" / "dashboard"

dashboard_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("===== DASHBOARD EXPORT LOCATION =====")
print(dashboard_dir)


# ------------------------------------------------------------
# PART 2: DEFINE TABLES TO EXPORT
# ------------------------------------------------------------

export_tables = {

    # Historical demand
    "reporting.v_airpods_weekly_demand":
        "dashboard_airpods_weekly_demand.csv",

    # Behavioral demand signals
    "reporting.v_headphone_daily_signals":
        "dashboard_headphone_daily_signals.csv",

    # Retail demand
    "reporting.v_retail_monthly_demand":
        "dashboard_retail_monthly_demand.csv",

    # Forecast model performance
    "reporting.v_forecast_model_performance":
        "dashboard_forecast_model_performance.csv",

    # Purchase intent models
    "reporting.v_purchase_intent_models":
        "dashboard_purchase_intent_models.csv",

    # Architecture and quality metadata
    "metadata.architecture_catalog":
        "dashboard_architecture_catalog.csv",

    "metadata.data_quality_report":
        "dashboard_data_quality_report.csv",

    "metadata.kpi_dictionary":
        "dashboard_kpi_dictionary.csv"
}


# ------------------------------------------------------------
# PART 3: CONNECT TO DUCKDB
# ------------------------------------------------------------

con = duckdb.connect(
    str(database_path),
    read_only=True
)

export_results = []

try:

    # --------------------------------------------------------
    # PART 4: EXPORT DATABASE RESULTS
    # --------------------------------------------------------

    for source_table, output_filename in export_tables.items():

        # Retrieve the complete SQL result
        result = con.execute(
            f"SELECT * FROM {source_table}"
        )

        rows = result.fetchall()

        columns = [
            column[0]
            for column in result.description
        ]

        # Convert to Pandas
        export_df = pd.DataFrame(
            rows,
            columns=columns
        )

        # Define output path
        output_path = dashboard_dir / output_filename

        # Export to CSV
        export_df.to_csv(
            output_path,
            index=False
        )

        # Reload exported file for validation
        verified_df = pd.read_csv(output_path)

        assert len(verified_df) == len(export_df), (
            f"Export row mismatch: {output_filename}"
        )

        assert list(verified_df.columns) == columns, (
            f"Export column mismatch: {output_filename}"
        )

        # Record export details
        export_results.append({
            "Source_Table": source_table,
            "Export_File": output_filename,
            "Rows": len(export_df),
            "Columns": len(columns),
            "Status": "PASS"
        })

        print(
            f"Exported: {output_filename} "
            f"({len(export_df)} rows)"
        )


    # --------------------------------------------------------
    # PART 5: VALIDATE EXPECTED ROW COUNTS
    # --------------------------------------------------------

    dashboard_export_report = pd.DataFrame(
        export_results
    )

    expected_export_rows = {
        "dashboard_airpods_weekly_demand.csv": 51,
        "dashboard_headphone_daily_signals.csv": 61,
        "dashboard_retail_monthly_demand.csv": 62,
        "dashboard_forecast_model_performance.csv": 8,
        "dashboard_purchase_intent_models.csv": 7,
        "dashboard_architecture_catalog.csv": 16,
        "dashboard_data_quality_report.csv": 16,
        "dashboard_kpi_dictionary.csv": 8
    }

    for record in export_results:

        expected = expected_export_rows[
            record["Export_File"]
        ]

        assert record["Rows"] == expected, (
            f"Unexpected row count: {record['Export_File']}"
        )


    # --------------------------------------------------------
    # PART 6: VERIFY CRITICAL QUALITY FLAGS
    # --------------------------------------------------------

    signals_df = pd.read_csv(
        dashboard_dir /
        "dashboard_headphone_daily_signals.csv"
    )

    retail_df = pd.read_csv(
        dashboard_dir /
        "dashboard_retail_monthly_demand.csv"
    )

    # November 15 purchase anomaly must remain excluded.
    anomaly = signals_df[
        signals_df["Event_Date"] == "2019-11-15"
    ]

    assert len(anomaly) == 1

    assert pd.isna(
        anomaly["Purchases_For_Analysis"].iloc[0]
    )

    # January and February 2021 remain unaudited.
    partial_2021 = retail_df[
        retail_df["Month"].isin([
            "2021-01-01",
            "2021-02-01"
        ])
    ]

    assert len(partial_2021) == 2

    assert partial_2021[
        [
            "Calendar_Days",
            "Days_With_Sales",
            "Recorded_Day_Pct"
        ]
    ].isna().all().all()

    print("\n===== DATA QUALITY PRESERVATION =====")

    print("November 15 anomaly preserved: PASS")
    print("2021 unaudited coverage preserved: PASS")


    # --------------------------------------------------------
    # PART 7: DISPLAY EXPORT VALIDATION
    # --------------------------------------------------------

    print("\n===== DASHBOARD EXPORT REPORT =====")

    display(dashboard_export_report)

    assert len(dashboard_export_report) == 8

    assert (
        dashboard_export_report["Status"] == "PASS"
    ).all()

    print("\n===== FINAL VALIDATION =====")
    print("Dashboard CSV files:", len(export_results))
    print("All exports validated:", True)

    print("\n===== STEP 7 COMPLETE =====")


finally:

    con.close()

    print("\nDatabase connection closed safely.")


===== DASHBOARD EXPORT LOCATION =====
/Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/dashboard
Exported: dashboard_airpods_weekly_demand.csv (51 rows)
Exported: dashboard_headphone_daily_signals.csv (61 rows)
Exported: dashboard_retail_monthly_demand.csv (62 rows)
Exported: dashboard_forecast_model_performance.csv (8 rows)
Exported: dashboard_purchase_intent_models.csv (7 rows)
Exported: dashboard_architecture_catalog.csv (16 rows)
Exported: dashboard_data_quality_report.csv (16 rows)
Exported: dashboard_kpi_dictionary.csv (8 rows)

===== DATA QUALITY PRESERVATION =====
November 15 anomaly preserved: PASS
2021 unaudited coverage preserved: PASS

===== DASHBOARD EXPORT REPORT =====


,Source_Table,Export_File,Rows,Columns,Status
0,reporting.v_airpods_weekly_demand,dashboard_airpods_weekly_demand.csv,51,2,PASS
1,reporting.v_headphone_daily_signals,dashboard_headphone_daily_signals.csv,61,7,PASS
2,reporting.v_retail_monthly_demand,dashboard_retail_monthly_demand.csv,62,8,PASS
3,reporting.v_forecast_model_performance,dashboard_forecast_model_performance.csv,8,6,PASS
4,reporting.v_purchase_intent_models,dashboard_purchase_intent_models.csv,7,7,PASS
5,metadata.architecture_catalog,dashboard_architecture_catalog.csv,16,5,PASS
6,metadata.data_quality_report,dashboard_data_quality_report.csv,16,9,PASS
7,metadata.kpi_dictionary,dashboard_kpi_dictionary.csv,8,4,PASS



===== FINAL VALIDATION =====
Dashboard CSV files: 8
All exports validated: True

===== STEP 7 COMPLETE =====

Database connection closed safely.



# Integrated Demand Intelligence Framework — Final Findings

## 1. Project Objective

Develop a centralized analytical reporting framework for
the Hardware Demand-Sensing & Launch Readiness Planner.

Consolidate processed analytical outputs from four
independent Kaggle datasets into a reusable SQL-based
reporting architecture.

The framework will support future demand forecasting,
inventory simulations, launch-readiness scenarios,
and executive dashboard development.

## 2. Analytical Data Sources

Four independent analytical modules were established:

1. Historical Demand Forecasting
   - Weekly AirPods units sold
   - Rolling forecast predictions
   - Model performance comparisons

2. eCommerce Behavior
   - Apple headphone product views
   - Add-to-cart and purchase events
   - Behavioral forecasting experiments

3. Retail Analytics
   - Historical units sold
   - Product category demand
   - Geographic and channel analysis
   - Monthly coverage-quality indicators

4. Purchase Intent
   - Classification model performance
   - Customer and product feature experiments
   - Confusion-matrix results

A total of 16 processed CSV files were cataloged.

These sources are analytically complementary but do not
share verified transaction-level or customer-level keys.

## 3. Data Quality Validation

Validated all 16 processed datasets for:
- Record counts
- Missing values
- Duplicate complete records
- Available historical date ranges

All 14 previously established row-count expectations
passed. Two additional files were inspected without
predefined expected counts.

Seven missing cells were investigated.

One missing purchase-analysis value represents the
intentionally excluded November 15, 2019 anomaly.

Six missing retail coverage values relate to the
unaudited January and February 2021 reporting periods.

No missing values were automatically replaced.

## 4. DuckDB Data Architecture

Created a persistent analytical database:

data/warehouse/demand_intelligence.duckdb

Database schemas:
- demand
- ecommerce
- retail
- intent
- metadata

Persistent tables:
- 16 analytical tables
- 3 metadata tables
- 19 total tables

Metadata tables contain:
- Dataset architecture definitions
- Data-quality summaries
- KPI definitions

CSV-to-SQL record counts were reconciled successfully.

## 5. SQL Reporting Layer

Created five reusable SQL views:

- reporting.v_airpods_weekly_demand
- reporting.v_headphone_daily_signals
- reporting.v_retail_monthly_demand
- reporting.v_forecast_model_performance
- reporting.v_purchase_intent_models

Each view was validated against its expected row count.

Source identifiers and measurement units were retained
to prevent misleading cross-dataset comparisons.

## 6. Key Analytical Findings

Historical AirPods demand:
- 51 observed weeks
- 15,364 units sold
- Average weekly demand: 301.25 units

Best initial AirPods forecasting baseline:
- Naive model
- WAPE: 8.60%

Apple headphone eCommerce behavior:
- 61 observed days
- 1,058,017 product-view events
- 109,366 cart events
- 43,377 recorded purchase events
- One flagged purchase-recording anomaly

Retail analysis:
- April 2019 recorded sales on only 7 of 30 days
- Recorded-day coverage was 23.33%
- Low reported sales should not automatically be
  interpreted as a genuine seasonal demand decline

Purchase-intent classification:
- Best exploratory ROC-AUC: 0.800
- Random Forest with satisfaction features
- Customer-enriched models without satisfaction
  achieved ROC-AUC values around 0.66

Satisfaction-enhanced model results require additional
assessment of potential information leakage.

## 7. Dashboard-Ready Data Exports

Exported eight validated CSV files to:

data/dashboard/

Contents:
- Historical weekly demand
- Daily behavioral demand signals
- Retail monthly demand
- Forecast model performance
- Purchase-intent model performance
- Architecture metadata
- Data-quality metadata
- KPI definitions

All eight exports passed row-count and column checks.

Known data-quality exceptions were preserved.

## 8. Business Applications

The reporting framework enables:
- Consistent historical demand monitoring
- Source-specific demand-signal analysis
- Forecast model benchmarking
- Retail demand and coverage reporting
- Purchase-intent research
- Preparation of inventory-planning inputs

It provides a foundation for decision-support
simulations but does not yet constitute a validated
operational forecasting or replenishment system.

## 9. Analytical Limitations

- The four datasets come from independent sources.
- Shared product/customer identities are unverified.
- Purchase events are not necessarily sales units.
- Customer purchase intent is not realized demand.
- Some historical periods have incomplete coverage.
- Forecasting performance requires further validation.
- Inventory and supplier inputs are not yet available.
- Future inventory scenarios will use clearly labeled
  assumptions where operational data is unavailable.

## 10. Next Development Phase

The next phase will focus on:

1. Strengthening time-ordered forecasting validation
2. Evaluating the incremental value of demand signals
3. Developing inventory and replenishment simulations
4. Modeling supplier delays and demand surges
5. Defining launch-readiness risk indicators
6. Building an executive Power BI dashboard
7. Preparing the final GitHub and website case study
